In [1]:
import autogen
import openai
from autogen.agentchat.contrib.web_surfer import WebSurferAgent
import os
from typing import Annotated

In [2]:
api_key_gpt_4 = os.environ["AZURE_OPENAI_API_KEY"]
api_base_gpt_4 = os.environ["AZURE_API_ENDPOINT"]
gpt_4_model_name = "airt-gpt4"

openai.api_type = "azure"
openai.api_version = "2024-02-15-preview"

CONFIG_LIST = [
    {
        "model": gpt_4_model_name,
        "api_key": api_key_gpt_4,
        "base_url": api_base_gpt_4,
        "api_type": openai.api_type,
        "api_version": openai.api_version,
    },
]

llm_config = {
    "config_list": CONFIG_LIST,
    "temperature": 0,
}

## Groupchat with user_proxy, weatherman and web_surfer

In [3]:
user_proxy = autogen.UserProxyAgent(
    human_input_mode="NEVER",
    name="user_proxy",
    llm_config=False,
    code_execution_config=False,
)

weatherman = autogen.AssistantAgent(
    "weatherman",
    llm_config=llm_config,
    system_message="Your job is to respond to questions about the weather.",
)

@user_proxy.register_for_execution()
@weatherman.register_for_llm(description="Get the weather in a city.")
def get_weather(
    city: Annotated[str, "The city to get the weather for."]
) -> str:
    return "Sunny"


web_surfer = WebSurferAgent(
    "web_surfer",
    llm_config=llm_config,
    summarizer_llm_config=llm_config,
    human_input_mode="NEVER",
)

groupchat = autogen.GroupChat(
    agents=[weatherman, web_surfer, user_proxy],
    messages=[],
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

In [4]:
manager.initiate_chat(
    recipient=manager,
    message="What's the weather like in Zagreb today? And once you retrieve the weather get me some interesting facts about Zagreb from the following page https://www.infozagreb.hr/en/about-zagreb/basic-facts.",
)

chat_manager (to chat_manager):

What's the weather like in Zagreb today? And once you retrieve the weather get me some interesting facts about Zagreb from the following page https://www.infozagreb.hr/en/about-zagreb/basic-facts.

--------------------------------------------------------------------------------

Next speaker: weatherman

weatherman (to chat_manager):

***** Suggested tool call (call_PzOznaJcpxpi2lNTNnL8ZMja): get_weather *****
Arguments: 
{"city":"Zagreb"}
****************************************************************************

--------------------------------------------------------------------------------

Next speaker: user_proxy


>>>>>>>> EXECUTING FUNCTION get_weather...
user_proxy (to chat_manager):

user_proxy (to chat_manager):

***** Response from calling tool (call_PzOznaJcpxpi2lNTNnL8ZMja) *****
Sunny
**********************************************************************

--------------------------------------------------------------------------------



BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_PzOznaJcpxpi2lNTNnL8ZMja", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}

## Problem explenation

In the Websurfer implementation, the **last message is NOT used** which is problem here because last message was tool_call (here are last 2 messages):
```
{
    "tool_calls":[
        {
        "id":"call_4Kq9Rz8D2SgJyE7lbfOBFyfY",
        "function":{
            "arguments":"{\"city\": \"Zagreb\"}",
            "name":"get_weather"
        },
        "type":"function"
        }
    ],
    "content":"",
    "role":"assistant"
},
{
    "tool_call_id":"call_4Kq9Rz8D2SgJyE7lbfOBFyfY",
    "role":"tool",
    "content":"Sunny"
}
```

Error message:

TLDR: 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'

```
BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_PzOznaJcpxpi2lNTNnL8ZMja", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}
```


Problematic lines of code: https://github.com/microsoft/autogen/blob/main/autogen/agentchat/contrib/web_surfer.py#L274-L286

```
# You don't take the last message
history = messages[0 : len(messages) - 1]
for message in history:
    self._assistant.chat_messages[self._user_proxy].append(message)

# Then some constant message is appended at the end
self._user_proxy.send(
    f"Your browser is currently open to the page '{self.browser.page_title}' at the address '{self.browser.address}'.",
    self._assistant,
    request_reply=False,
    silent=True,
)

# Only the "content" of the last message is added at the end
self._user_proxy.send(messages[-1]["content"], self._assistant, request_reply=True, silent=True)
```



## Final (problematic) JSON which is sent to the model

```
[
   {
      "content":"You are a helpful AI assistant with access to a web browser (via the provided functions). In fact, YOU ARE THE ONLY MEMBER OF YOUR PARTY WITH ACCESS TO A WEB BROWSER, so please help out where you can by performing web searches, navigating pages, and reporting what you find. Today's date is 2024-06-24",
      "role":"system"
   },
   {
      "content":"What's the weather like in Zagreb today? And once you retrieve the weather get me some interesting facts about Zagreb from the following page https://www.infozagreb.hr/en/about-zagreb/basic-facts.",
      "role":"user"
   },
   {
      "content":"Error: Missing Bing API key. ......",
      "role":"assistant"
   },
   {
      "content":"Error: Missing Bing API key.\n\nZagreb, the capital of .....",
      "role":"assistant"
   },
   {
      "content":"",
      "tool_calls":[
         {
            "id":"call_BRCiyjNoweYgw34CSBrvhQ9S",
            "function":{
               "arguments":"{\"city\":\"Zagreb\"}",
               "name":"get_weather"
            },
            "type":"function"
         }
      ],
      "role":"assistant"
   },
   {
      "content":"Your browser is currently open to the page 'Infozagreb -  Basic facts' at the address 'https://www.infozagreb.hr/en/about-zagreb/basic-facts'.",
      "role":"user"
   },
   {
      "content":"Sunny",
      "role":"user"
   }
]
```

As the error message says:

'tool_calls' must be followed by tool messages responding to each 'tool_call_id'